In [1]:
import kagglehub
path = kagglehub.dataset_download("emmarex/plantdisease")
print("Path to dataset files:", path)
dataset_path = path + "/PlantVillage"

Using Colab cache for faster access to the 'plantdisease' dataset.
Path to dataset files: /kaggle/input/plantdisease


In [2]:
!pip install tensorflow
import tensorflow as tf
from tensorflow.keras.layers import Dense,GlobalAveragePooling2D
from tensorflow.keras import Model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [3]:
base_model = ResNet50(
    include_top=False,
    weights="imagenet",
    input_shape=(224,224,3)
)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [4]:
for layer in base_model.layers:
  layer.trainable = False

In [5]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
output = Dense(15, activation='softmax')(x)

In [6]:
img_model = Model(inputs=base_model.inputs, outputs=output)

In [7]:
img_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [8]:
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

In [9]:
train_data = train_gen.flow_from_directory(
    dataset_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)
val_data = train_gen.flow_from_directory(
    dataset_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

Found 16516 images belonging to 15 classes.
Found 4122 images belonging to 15 classes.


In [10]:
history = img_model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
517/517 ━━━━━━━━━━━━━━━━━━━━ 375s 693ms/step - accuracy: 0.1957 - loss: 2.4270 - val_accuracy: 0.2555 - val_loss: 2.2906
Epoch 2/10
517/517 ━━━━━━━━━━━━━━━━━━━━ 268s 518ms/step - accuracy: 0.2747 - loss: 2.2253 - val_accuracy: 0.2737 - val_loss: 2.2840
Epoch 3/10
517/517 ━━━━━━━━━━━━━━━━━━━━ 270s 522ms/step - accuracy: 0.3357 - loss: 2.0485 - val_accuracy: 0.3651 - val_loss: 1.9452
Epoch 4/10
517/517 ━━━━━━━━━━━━━━━━━━━━ 269s 519ms/step - accuracy: 0.3709 - loss: 1.9171 - val_accuracy: 0.4059 - val_loss: 1.8494
Epoch 5/10
517/517 ━━━━━━━━━━━━━━━━━━━━ 268s 517ms/step - accuracy: 0.4006 - loss: 1.8228 - val_accuracy: 0.4447 - val_loss: 1.7737
Epoch 6/10
517/517 ━━━━━━━━━━━━━━━━━━━━ 264s 512ms/step - accuracy: 0.4268 - loss: 1.7442 - val_accuracy: 0.4364 - val_loss: 1.7007
Epoch 7/10
517/517 ━━━━━━━━━━━━━━━━━━━━ 267s 516ms/step - accuracy: 0.4444 - loss: 1.6935 - val_accuracy: 0.4447 - val_loss: 1.6765
Epoch 8/10
517/517 ━━━━━━━━━━━━━━━━━━━━ 266s 515ms/step - accuracy: 0.4508 -

In [11]:
img_model.save("plant_disease_model.h5")